# Lab: Staggered adoption and TWFE diagnostics (Python)

[View this lab on the QED Labs website](https://defenceeconomist.github.io/qedlabs/labs/difference-in-differences-staggered-diagnostics-python-lab.html)

## How to use this lab

Allow 45–60 minutes. Basic regression and Python data-frame familiarity are assumed.
Use the [tested environment setup](https://defenceeconomist.github.io/qedlabs/labs/difference-in-differences-reproducibility.html) before running every cell in order.

[R companion](https://defenceeconomist.github.io/qedlabs/labs/difference-in-differences-staggered-diagnostics-lab.html) · [Application catalogue](https://defenceeconomist.github.io/qedlabs/notes/did/difference-in-differences-applications.html)

The code downloads a checksum-verified upstream data file on first use and caches it locally. This is a teaching reproduction, not a replication of every specification in the original paper.

## Research question

The castle-doctrine application relates staggered changes in state self-defense law to log homicide [@cheng2013castle]. This exercise uses the same unweighted sample as the R lab; it does not claim to reproduce the paper's full preferred specification.

## 1. Reconstruct adoption and support

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyreadr

# Immutable upstream package data; do not silently accept a changed file.
url = "https://raw.githubusercontent.com/cran/causaldata/6d25e70297812a2e89a1a20e3bc24b32f6d3fbaf/data/castle.rda"
expected_sha256 = "d7e50add4a642a1320f254e96cab436a19d0f2d9f0434632c511a8f05de8a355"
cache = Path(".qedlabs-cache")
cache.mkdir(exist_ok=True)
path = cache / "castle.rda"
if not path.exists():
    payload = urlopen(url, timeout=60).read()
    assert hashlib.sha256(payload).hexdigest() == expected_sha256, "Data checksum mismatch"
    path.write_bytes(payload)
assert hashlib.sha256(path.read_bytes()).hexdigest() == expected_sha256, "Cached data changed"
data = pyreadr.read_r(str(path))["castle"]
print({p: metadata.version(p) for p in ["pandas", "numpy", "pyreadr"]})
print(data.shape)

In [ ]:
import pyfixest as pf
panel = data[['sid', 'year', 'post', 'l_homicide']].copy().sort_values(['sid', 'year'])
assert not panel.duplicated(['sid', 'year']).any()
assert panel.notna().all().all()
assert panel.groupby('sid').size().nunique() == 1
assert panel.groupby('sid').post.diff().dropna().ge(0).all()
first = panel.loc[panel.post == 1].groupby('sid').year.min()
panel['first_treat'] = panel.sid.map(first).fillna(0).astype(int)
panel['event_time'] = np.where(panel.first_treat > 0, panel.year - panel.first_treat, -1000).astype(int)
assert panel.first_treat.eq(0).any()
print(panel.groupby('first_treat').sid.nunique())
support = panel.assign(never=panel.first_treat.eq(0), future=panel.first_treat.gt(panel.year)).groupby('year')[['never', 'future']].sum()
print(support)
print({'states': panel.sid.nunique(), 'years': panel.year.nunique(), 'rows': len(panel)})

In [ ]:
adoption = panel.pivot(index='sid', columns='year', values='post')
plt.figure(figsize=(9, 6)); plt.imshow(adoption, aspect='auto', interpolation='nearest', cmap='Blues')
plt.xticks(range(len(adoption.columns)), adoption.columns, rotation=45)
plt.xlabel('Year'); plt.ylabel('States (one row per state)'); plt.colorbar(label='Treated'); plt.show()
panel.groupby(['first_treat', 'year']).l_homicide.mean().unstack(0).plot(figsize=(9, 4))
plt.ylabel('Mean log homicide'); plt.legend(title='First treatment (0 = never)'); plt.show()

Checkpoint: which controls disappear as potential untreated comparisons after adopting? Does a never-treated label establish institutional comparability?

## 2. Static TWFE and conventional event study

In [ ]:
twfe = pf.feols('l_homicide ~ post | sid + year', data=panel, vcov={'CRV1': 'sid'})
# Never-treated units have zeros in every event indicator; -1 is the reference.
event_columns = []
for event in sorted(panel.loc[panel.first_treat > 0, 'event_time'].unique()):
    if event == -1:
        continue
    name = f"event_{'m' if event < 0 else 'p'}{abs(event)}"
    panel[name] = ((panel.first_treat > 0) & (panel.event_time == event)).astype(int)
    event_columns.append((int(event), name))
conventional = pf.feols('l_homicide ~ ' + ' + '.join(n for _, n in event_columns) + ' | sid + year',
                       data=panel, vcov={'CRV1': 'sid'})
print(twfe.tidy())

State clustering allows within-state dependence; it does not correct biased comparisons. Conventional TWFE can use already-treated states as controls when treatment effects change with exposure [@goodmanbacon2021timing].

## 3. Cohort-aware event study

In [ ]:
saturated = pf.event_study(panel, yname='l_homicide', idname='sid', tname='year',
                           gname='first_treat', estimator='saturated', cluster='sid')
sa = saturated.aggregate().astype(float)
sa.index = sa.index.astype(float).astype(int)
con = pd.DataFrame({'event_time': [e for e, n in event_columns],
                    'estimate': [conventional.coef()[n] for e, n in event_columns],
                    'se': [conventional.se()[n] for e, n in event_columns]}).set_index('event_time')
fig, ax = plt.subplots(figsize=(9, 5))
for label, table in [('Conventional TWFE', con),
                     ('Cohort-aware', sa.rename(columns={'Estimate': 'estimate', 'Std. Error': 'se'}))]:
    visible = table.loc[(table.index >= -5) & (table.index <= 4)]
    assert np.isfinite(visible[['estimate', 'se']]).all().all()
    ax.errorbar(visible.index, visible.estimate, yerr=1.96 * visible.se, marker='o', label=label)
ax.axhline(0, color='grey'); ax.axvline(-1, color='grey', linestyle='--')
ax.set(xlabel='Years relative to adoption (-1 omitted)', ylabel='Log-homicide effect')
ax.legend(); plt.show()
print(sa.loc[-5:4])

All available event periods enter the regressions; only the display is restricted to −5 through +4. Bands here are pointwise normal approximations, not simultaneous bands. The pinned PyFixest implementation labels saturated event studies experimental; the validation compares its point estimates with `fixest::sunab()` on this sample.

## R extension: Bacon decomposition

Run the decomposition in the [R companion](https://defenceeconomist.github.io/qedlabs/labs/difference-in-differences-staggered-diagnostics-lab.html#step-5-decompose-the-twfe-coefficient). The weighted component estimates must reconstruct the static TWFE coefficient. The decomposition's nonnegative weights on 2×2 comparisons must not be confused with potentially negative weights on underlying treatment effects.

Interpretation exercise: a positive weight on a later-versus-earlier contrast does not make the earlier cohort an untreated control. If its treatment effect grows during that comparison, the contrast subtracts that growth.

## Worked answers and reporting exercise

- **Shrinking support:** future adopters cease to be untreated controls once exposed. Check which cohorts contribute at each event time.
- **Why coefficients differ:** conventional leads/lags can mix effects from other event periods; cohort-aware estimates change the comparisons and weights [@sunabraham2021dynamic].
- **What a lead detects:** it may reveal a design failure; a nonsignificant estimate may simply be imprecise. Pre-testing cannot certify parallel trends.
- **What to report:** treatment coding, unweighted sample, state clustering, event reference, comparison rule, and unsupported horizons.

Write a short design assessment separating coding checks from claims about policy assignment. Connect it to the [Goodman-Bacon notes](https://defenceeconomist.github.io/qedlabs/notes/did/goodman-bacon-treatment-timing-notes.html).